# HarborMind Data Pipeline - Phase 1: Data Ingestion (Bronze Layer)

**Objective:**
This notebook handles the sequential downloading, decompression, and initial spatial filtering of heavy NOAA AIS datasets. 

**Key Engineering Constraints Addressed:**
- **Out-of-Memory (OOM) Prevention:** Raw `.csv.zst` files are heavily compressed. Attempting to load multiple days simultaneously on a trial Microsoft Fabric cluster causes OOM errors. This notebook is designed to run in a sequential micro-batch (day-by-day) to maintain memory stability.
- **Geofencing:** Only data within UTM Zone 11 (Port of LA/LB) is retained to minimize storage costs.

**Author:** Kelvin Nguyen
**Architecture:** Medallion (Raw -> Bronze)


## 1. Setup & Environment
Define target dates and temporary storage paths on OneLake.

In [ ]:
target_date = "2023-06-01"
batch_name = "2023H2"


## 2. Extraction: Download & Decompress
Downloads the heavy `.csv.zst` from NOAA servers and uses `zstandard` to decompress it locally on the Spark driver before loading it into memory.


In [ ]:
import os
import requests
import zstandard as zstd
from pyspark.sql import functions as F
from pyspark.sql import Window

In [ ]:


year = target_date[:4]
url = f"https://noaaocm.blob.core.windows.net/ais/csv2/csv{year}/ais-{target_date}.csv.zst"

# Paths
python_path = "/lakehouse/default/Files/Bronze/raw_csvs/"
os.makedirs(python_path, exist_ok=True)
compressed = f"{python_path}ais-{target_date}.csv.zst"
csv_file_python = f"{python_path}ais-{target_date}.csv"

# Download
print(f"Downloading {target_date}...")
resp = requests.get(url, stream=True)
resp.raise_for_status()
with open(compressed, 'wb') as f:
    for chunk in resp.iter_content(chunk_size=1024*1024):
        f.write(chunk)
print(f"Downloaded: {os.path.getsize(compressed) / 1e6:.1f} MB")

# Decompress
print(f"Decompressing...")
with open(compressed, 'rb') as src:
    dctx = zstd.ZstdDecompressor()
    with open(csv_file_python, 'wb') as dst:
        dctx.copy_stream(src, dst)

# Delete compressed
os.remove(compressed)
print(f"Ready: {os.path.getsize(csv_file_python) / 1e9:.2f} GB")



## 3. Transformation: Spatial Filtering (Geofencing)
Load the raw CSV into a PySpark DataFrame, enforce schema types, and aggressively filter out any vessels not within the Southern California bounding box (Latitude: 33.5 to 33.9, Longitude: -118.5 to -118.0) or not classified as Cargo/Tankers (Types 70-89).


In [ ]:
csv_file_spark = f"Files/Bronze/raw_csvs/ais-{target_date}.csv"
df = spark.read.csv(csv_file_spark, header=True, inferSchema=True)

# Rename
column_map = {
    "MMSI": "mmsi", "BaseDateTime": "base_date_time",
    "LAT": "latitude", "LON": "longitude",
    "SOG": "sog", "COG": "cog", "Heading": "heading",
    "VesselName": "vessel_name", "IMO": "imo",
    "CallSign": "call_sign", "VesselType": "vessel_type",
    "Status": "status", "Length": "length", "Width": "width",
    "Draft": "draft", "Cargo": "cargo",
    "TransceiverClass": "transceiver"
}
for old, new in column_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

# Cast types
df = df.withColumn("base_date_time", F.to_timestamp("base_date_time"))
df = df.withColumn("vessel_type", F.col("vessel_type").cast("integer"))
df = df.withColumn("latitude", F.col("latitude").cast("double"))
df = df.withColumn("longitude", F.col("longitude").cast("double"))

# Filter and deduplicate Zone 11
df_zone11 = df.filter(
    (F.col("latitude").between(33.5, 33.9)) & 
    (F.col("longitude").between(-118.5, -118.0)) &
    (F.col("vessel_type").between(70, 89))
)
df_zone11 = df_zone11.dropDuplicates(["mmsi", "base_date_time"])
# ADD batch column to track
df_zone11 = df_zone11.withColumn("batch", F.lit(batch_name))
count = df_zone11.count()
print(f"🎯 {target_date}: {count} rows")


In [ ]:
if count > 0:
    df_zone11.write.mode("append").format("delta") \
        .saveAsTable("ais_zone11_bronze")
    print(f"✅ Appended to Bronze")

os.remove(csv_file_python)
print(f"🗑️ Cleaned up")

## 5. Post-Ingestion Data Validation (Sanity Checks)
After appending the micro-batch, we run automated validation checks to ensure data integrity. This confirms we ingested the correct target date and successfully filtered the required vessel types within the geofenced Zone 11.

In [ ]:
# # ===== 1. CHECK BRONZE TABLE =====
# df_bronze = spark.read.format("delta").table("ais_zone11_bronze")

# print(f"Total rows: {df_bronze.count()}")
# print(f"Columns: {df_bronze.columns}")
# print(f"Distinct dates: {df_bronze.select(F.to_date('base_date_time').alias('date')).distinct().count()}")

# # Phải ra 5 dates
# df_bronze.select(F.to_date("base_date_time").alias("date")) \
#     .distinct().orderBy("date").show()

In [ ]:
# from pyspark.sql import functions as F

# # ===== BRONZE =====
# df_bronze = spark.read.format("delta").table("ais_zone11_bronze")

# print("=" * 60)
# print("BRONZE LAYER")
# print("=" * 60)
# print(f"Rows: {df_bronze.count()}")
# print(f"Columns: {len(df_bronze.columns)}")
# print(f"Columns: {df_bronze.columns}\n")

# df_bronze.select(
#     F.min("latitude").alias("min_lat"), F.max("latitude").alias("max_lat"),
#     F.min("longitude").alias("min_lon"), F.max("longitude").alias("max_lon"),
#     F.min("vessel_type").alias("min_vtype"), F.max("vessel_type").alias("max_vtype"),
#     F.min("base_date_time").alias("from"), F.max("base_date_time").alias("to"),
#     F.countDistinct("mmsi").alias("unique_ships")
# ).show(truncate=False)

# df_bronze.groupBy(F.to_date("base_date_time").alias("date")) \
#     .agg(F.count("*").alias("rows"), F.countDistinct("mmsi").alias("ships")) \
#     .orderBy("date").show()

# # ===== GOLD =====
# print("=" * 60)
# print("GOLD LAYER")
# print("=" * 60)

# try:
#     df_gold = spark.read.parquet("Files/Gold/ais_zone11_2023H2_preprocessed.parquet")
    
#     print(f"Rows: {df_gold.count()}")
#     print(f"Columns: {len(df_gold.columns)}")
#     print(f"Columns: {df_gold.columns}\n")

#     # Null check
#     total = df_gold.count()
#     print("Null counts:")
#     for col_name in df_gold.columns:
#         nulls = df_gold.filter(F.col(col_name).isNull()).count()
#         if nulls > 0:
#             print(f"  {col_name}: {nulls} ({round(nulls/total*100,1)}%)")

#     # Feature ranges
#     print("\nFeature ranges:")
#     df_gold.select(
#         F.min("sog").alias("min_sog"), F.max("sog").alias("max_sog"),
#         F.min("acceleration").alias("min_accel"), F.max("acceleration").alias("max_accel"),
#         F.min("distance_to_port").alias("min_dist"), F.max("distance_to_port").alias("max_dist"),
#         F.min("ship_density").alias("min_density"), F.max("ship_density").alias("max_density"),
#         F.min("delay_minutes").alias("min_delay"), F.max("delay_minutes").alias("max_delay"),
#     ).show(truncate=False)

#     # Visits
#     visits = df_gold.select("mmsi", "visit_id").distinct().count()
#     moored = df_gold.filter(F.col("delay_minutes").isNotNull()) \
#         .select("mmsi", "visit_id").distinct().count()
#     print(f"Visits: {visits} total, {moored} moored ({round(moored/visits*100,1)}%)")

#     # Sample
#     print("\nSample data:")
#     df_gold.select("mmsi", "base_date_time", "sog", "distance_to_port", 
#                    "acceleration", "ship_density", "delay_minutes") \
#         .orderBy("base_date_time").show(10, truncate=False)

# except Exception as e:
#     print(f"Gold has not existed: {e}")
#     print("→ Needing to run AIS_Preprocessing notebook first!")


In [ ]:
# df_gold = spark.read.parquet("Files/Gold/ais_zone11_2023H2_preprocessed.parquet")


# df_check = df_gold.filter(F.col("acceleration").isNotNull())

# w = Window.partitionBy("mmsi").orderBy("base_date_time")
# df_check = df_check.withColumn("td", 
#     F.col("base_date_time").cast("long") - F.lag("base_date_time",1).over(w).cast("long"))

# df_check.select(
#     F.min("td").alias("min_time_diff"),
#     F.max("td").alias("max_time_diff"),
#     F.avg("td").alias("avg_time_diff"),
#     F.count(F.when(F.col("td") < 60, 1)).alias("rows_under_60s"),
#     F.count(F.when(F.col("td") >= 60, 1)).alias("rows_over_60s"),
# ).show(truncate=False)


In [ ]:
# from pyspark.sql import functions as F
# from pyspark.sql import Window

# df = spark.read.parquet("Files/Gold/ais_zone11_2023H2_preprocessed.parquet")

# print("=" * 60)
# print("DATASET OVERVIEW")
# print("=" * 60)
# print(f"Rows:    {df.count():,}")
# print(f"Columns: {len(df.columns)}")
# print(f"Schema:")
# for col_name, dtype in df.dtypes:
#     print(f"  {col_name}: {dtype}")


In [ ]:
# print("=" * 60)
# print("DAILY VOLUME")
# print("=" * 60)

# daily = df.groupBy(to_date("base_date_time").alias("date")) \
#     .agg(
#         F.count("*").alias("rows"),
#         F.countDistinct("mmsi").alias("ships"),
#         F.avg("sog").alias("avg_sog"),
#         F.avg("ship_density").alias("avg_density")
#     ).orderBy("date")

# # Monthly summary
# monthly = df.withColumn("month", F.month("base_date_time")) \
#     .groupBy("month") \
#     .agg(
#         F.count("*").alias("rows"),
#         F.countDistinct("mmsi").alias("unique_ships"),
#         F.countDistinct(to_date("base_date_time")).alias("days"),
#         F.avg("sog").alias("avg_sog")
#     ).orderBy("month")

# print("Monthly summary:")
# monthly.show()

# # Weekday vs Weekend
# print("Weekday vs Weekend:")
# df.withColumn("is_wknd", F.dayofweek("base_date_time").isin([1,7])) \
#     .groupBy("is_wknd") \
#     .agg(F.avg("ship_density").alias("avg_density"),
#          F.avg("sog").alias("avg_sog"),
#          F.count("*").alias("rows")) \
#     .show()


In [ ]:
# print("=" * 60)
# print("FEATURE DISTRIBUTIONS")
# print("=" * 60)

# # Numeric features
# numeric_cols = ["sog", "cog", "heading", "draft", "length", "width",
#                 "vessel_area", "dimension_ratio", "draft_to_length_ratio",
#                 "distance_to_port", "heading_error", "acceleration",
#                 "time_in_zone_hours", "ship_density", "avg_port_speed",
#                 "port_throughput", "delay_minutes"]

# for col_name in numeric_cols:
#     stats = df.filter(F.col(col_name).isNotNull()).select(
#         F.count(col_name).alias("count"),
#         F.min(col_name).alias("min"),
#         F.percentile_approx(col_name, 0.25).alias("Q1"),
#         F.percentile_approx(col_name, 0.50).alias("median"),
#         F.percentile_approx(col_name, 0.75).alias("Q3"),
#         F.max(col_name).alias("max"),
#         F.avg(col_name).alias("mean"),
#         F.stddev(col_name).alias("std")
#     ).collect()[0]
    
#     print(f"\n{col_name}:")
#     print(f"  count={stats['count']:,}  min={stats['min']:.2f}  Q1={stats['Q1']:.2f}  "
#           f"median={stats['median']:.2f}  Q3={stats['Q3']:.2f}  max={stats['max']:.2f}  "
#           f"mean={stats['mean']:.2f}  std={stats['std']:.2f}")


In [ ]:
# print("=" * 60)
# print("NULL ANALYSIS")
# print("=" * 60)

# total = df.count()
# print(f"Total rows: {total:,}\n")

# for col_name in df.columns:
#     nulls = df.filter(F.col(col_name).isNull()).count()
#     pct = round(nulls / total * 100, 2)
#     status = "✅" if pct < 1 else "⚠️" if pct < 5 else "❌"
#     if nulls > 0:
#         print(f"  {status} {col_name}: {nulls:,} nulls ({pct}%)")

# print(f"\nColumns with 0 nulls: "
#       f"{sum(1 for c in df.columns if df.filter(F.col(c).isNull()).count() == 0)}/{len(df.columns)}")


In [ ]:
# print("=" * 60)
# print("VISIT & DELAY ANALYSIS")
# print("=" * 60)

# # Visit stats
# visits = df.select("mmsi", "visit_id").distinct()
# total_visits = visits.count()

# moored = df.filter(F.col("delay_minutes").isNotNull()) \
#     .select("mmsi", "visit_id").distinct()
# moored_visits = moored.count()

# print(f"Total visits: {total_visits}")
# print(f"Moored:       {moored_visits} ({round(moored_visits/total_visits*100,1)}%)")
# print(f"Never moored: {total_visits - moored_visits} ({round((total_visits-moored_visits)/total_visits*100,1)}%)")

# # Delay distribution (moored visits only)
# delay_stats = df.filter(F.col("delay_minutes").isNotNull()) \
#     .select("mmsi", "visit_id", "delay_minutes").distinct()

# print(f"\nDelay distribution (minutes):")
# delay_stats.select(
#     F.count("*").alias("visits"),
#     F.min("delay_minutes").alias("min"),
#     F.percentile_approx("delay_minutes", 0.25).alias("Q1"),
#     F.percentile_approx("delay_minutes", 0.50).alias("median"),
#     F.percentile_approx("delay_minutes", 0.75).alias("Q3"),
#     F.percentile_approx("delay_minutes", 0.95).alias("P95"),
#     F.max("delay_minutes").alias("max"),
#     F.avg("delay_minutes").alias("mean")
# ).show(truncate=False)

# # Delay categories
# print("Delay categories:")
# delay_stats.withColumn("category",
#     F.when(F.col("delay_minutes") == 0, "0: Immediate")
#     .when(F.col("delay_minutes") <= 60, "1: < 1 hour")
#     .when(F.col("delay_minutes") <= 360, "2: 1-6 hours")
#     .when(F.col("delay_minutes") <= 1440, "3: 6-24 hours")
#     .otherwise("4: > 24 hours")
# ).groupBy("category").count().orderBy("category").show()


In [ ]:
# print("=" * 60)
# print("VESSEL TYPE ANALYSIS")
# print("=" * 60)

# df.groupBy("vessel_type") \
#     .agg(
#         F.countDistinct("mmsi").alias("ships"),
#         F.count("*").alias("pings"),
#         F.avg("sog").alias("avg_sog"),
#         F.avg("length").alias("avg_length"),
#         F.avg("draft").alias("avg_draft")
#     ).orderBy("vessel_type").show()


In [ ]:
# print("=" * 60)
# print("SANITY CHECKS")
# print("=" * 60)

# total = df.count()
# checks = []

# # 1. Lat/Lon bounds
# lat_ok = df.filter((F.col("latitude") < 33.5) | (F.col("latitude") > 33.9)).count() == 0
# checks.append(("Zone 11 latitude bounds", lat_ok))

# lon_ok = df.filter((F.col("longitude") < -118.5) | (F.col("longitude") > -118.0)).count() == 0
# checks.append(("Zone 11 longitude bounds", lon_ok))

# # 2. Vessel types
# vtype_ok = df.filter(~F.col("vessel_type").between(70, 89)).count() == 0
# checks.append(("Vessel type 70-89 only", vtype_ok))

# # 3. No duplicate rows
# dedup_count = df.dropDuplicates(["mmsi", "base_date_time"]).count()
# checks.append(("No duplicates", dedup_count == total))

# # 4. Acceleration filter
# accel_check = df.filter(F.col("acceleration").isNotNull())
# w = Window.partitionBy("mmsi").orderBy("base_date_time")
# td = accel_check.withColumn("td", F.col("base_date_time").cast("long") - F.lag("base_date_time",1).over(w).cast("long"))
# bad_accel = td.filter((F.col("td").isNotNull()) & (F.col("td") < 60)).count()
# checks.append(("Acceleration time_diff >= 60s", bad_accel == 0))

# # 5. time_in_zone_hours >= 0
# tiz_ok = df.filter(F.col("time_in_zone_hours") < 0).count() == 0
# checks.append(("time_in_zone_hours >= 0", tiz_ok))

# # 6. distance_to_port > 0
# dist_ok = df.filter(F.col("distance_to_port") <= 0).count() == 0
# checks.append(("distance_to_port > 0", dist_ok))

# # 7. ship_density >= 0
# dens_ok = df.filter(F.col("ship_density") < 0).count() == 0
# checks.append(("ship_density >= 0", dens_ok))

# # 8. Enough data
# data_ok = total > 1_000_000
# checks.append((f"Sufficient data (>1M rows, got {total:,})", data_ok))

# # Results
# passed = sum(1 for _, ok in checks if ok)
# print(f"\nResults: {passed}/{len(checks)} passed\n")
# for name, ok in checks:
#     print(f"  {'GOOD | ' if ok else 'NOT GOOD | '} {name}")


In [ ]:
# from pyspark.sql import functions as F
# from pyspark.sql import Window

# df = spark.read.parquet("Files/Gold/ais_zone11_2023H2_preprocessed.parquet")
# total = df.count()

# print("=" * 60)
# print("OVERVIEW")
# print("=" * 60)
# print(f"Rows: {total:,}")
# print(f"Columns: {len(df.columns)}")

# # Date coverage
# date_stats = df.select(
#     F.min("base_date_time").alias("first"),
#     F.max("base_date_time").alias("last"),
#     F.countDistinct(F.to_date("base_date_time")).alias("days")
# ).collect()[0]
# print(f"Range: {date_stats['first']} → {date_stats['last']}")
# print(f"Days: {date_stats['days']} (expected: 92)")

# # Nulls
# print(f"\nNull counts:")
# for c in df.columns:
#     n = df.filter(F.col(c).isNull()).count()
#     if n > 0:
#         print(f"  {c}: {n:,} ({round(n/total*100,1)}%)")

# print(f"\n{'='*60}")
# print("KEY METRICS (SOG validation + smoothing + robust mooring)")
# print(f"{'='*60}")
# df.select(
#     F.min("sog").alias("min_sog"),
#     F.max("sog").alias("max_sog"),
#     F.min("acceleration").alias("min_accel"),
#     F.max("acceleration").alias("max_accel"),
#     F.avg("acceleration").alias("mean_accel"),
#     F.stddev("acceleration").alias("std_accel")
# ).show(truncate=False)

# # Visit + delay
# visits = df.select("mmsi", "visit_id").distinct().count()
# moored = df.filter(F.col("delay_minutes").isNotNull()) \
#     .select("mmsi", "visit_id").distinct().count()
# print(f"Visits: {visits}, Moored: {moored} ({round(moored/visits*100,1)}%)")

# # Delay distribution
# df.filter(F.col("delay_minutes").isNotNull()) \
#     .select("mmsi", "visit_id", "delay_minutes").distinct() \
#     .select(
#         F.count("*").alias("visits"),
#         F.min("delay_minutes").alias("min"),
#         F.percentile_approx("delay_minutes", 0.5).alias("median"),
#         F.max("delay_minutes").alias("max"),
#         F.avg("delay_minutes").alias("mean")
#     ).show(truncate=False)

# # SANITY CHECKS
# print(f"{'='*60}")
# print("SANITY CHECKS")
# print(f"{'='*60}")
# checks = []
# checks.append(("SOG max ≤ 30", df.agg(F.max("sog")).collect()[0][0] <= 30))
# checks.append(("Lat bounds", df.filter((F.col("latitude") < 33.5) | (F.col("latitude") > 33.9)).count() == 0))
# checks.append(("Lon bounds", df.filter((F.col("longitude") < -118.5) | (F.col("longitude") > -118.0)).count() == 0))
# checks.append(("Vessel type 70-89", df.filter(~F.col("vessel_type").between(70,89)).count() == 0))
# checks.append(("No duplicates", df.dropDuplicates(["mmsi","base_date_time"]).count() == total))

# for name, ok in checks:
#     print(f"  {'GOOD | ' if ok else 'NOT GOOD | '} {name}")
# print(f"\n{sum(1 for _,ok in checks if ok)}/{len(checks)} passed")


In [ ]:
# df = spark.read.format("delta").table("ais_zone11_bronze")

# print(f"Total rows: {df.count():,}")
# print(f"Date range: {df.agg(F.min('base_date_time'), F.max('base_date_time')).collect()[0]}")
# print(f"Unique days: {df.select(F.to_date('base_date_time').alias('d')).distinct().count()}")

# # Check missing days
# from pyspark.sql.functions import to_date, explode, sequence, col
# all_days = df.select(to_date("base_date_time").alias("d")).distinct().orderBy("d")
# all_days.show(200, truncate=False)  # xem list ngày có data

# # Daily row count — spot check
# df.groupBy(to_date("base_date_time").alias("date")) \
#   .agg(F.count("*").alias("rows"), F.countDistinct("mmsi").alias("ships")) \
#   .orderBy("date") \
#   .show(200)


## 6. Pipeline Teardown / Table Reset (Optional)
This section is strictly for administrative or debugging purposes. If the pipeline needs a fresh start (e.g., corrupted batch or restarting the ingestion process from scratch), uncomment and run the following cell to completely drop the Bronze table.
**WARNING: Destructive Action.**

In [ ]:
# WARNING: Running this will drop the entire Bronze Delta Table!
# spark.sql("DROP TABLE IF EXISTS ais_zone11_bronze")
# print("Bronze table has been successfully dropped.")

---

### Author's Disclaimer & Execution Note

**1. Microsoft Fabric Trial Expiration**

The entire Data Engineering pipeline (including the 50-hour sequential ingestion and all PySpark transformations) was successfully designed, executed, and validated using a Microsoft Fabric Trial account. 

As of this documentation's release, my trial period has expired. Therefore, I am unable to re-run the notebook to capture the live cell outputs for this presentation.

**2. Regarding Sanity Checks**

The Sanity Checks and Validation blocks provided above are **optional**. They were utilized during active development to rigorously verify the workflow and mathematical transformations. 

If you are cloning this repository to run on your own Spark/Fabric cluster:

- You can safely skip or delete the sanity check cells if you encounter any environment-specific syntax errors.
- The core ETL and Feature Engineering logic will function perfectly without them.
- If you run into issues reproducing this on different environments, feel free to ask an AI assistant (like ChatGPT or GitHub Copilot) to adapt the specific PySpark syntax to your current setup.

---
